# PCB defect visualiser

Compare a flawed PCB image (with annotated bounding boxes) against its clean reference.

Set `SAMPLE` below to either:
- a full path, e.g. `PCB_DATASET/images/Missing_hole/01_missing_hole_01.jpg`
- a bare stem, e.g. `01_missing_hole_01`
- `None` to pick a random example

In [ ]:
SAMPLE = None  # e.g. "01_missing_hole_01" or a full path; None = random

In [ ]:
import random
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

ROOT = Path.cwd()
DATASET = ROOT / "PCB_DATASET"
IMAGES_DIR = DATASET / "images"
ANNOT_DIR = DATASET / "Annotations"
CLEAN_DIR = DATASET / "PCB_USED"

In [ ]:
def resolve_sample(arg):
    if arg is None:
        imgs = list(IMAGES_DIR.rglob("*.jpg")) + list(IMAGES_DIR.rglob("*.JPG"))
        if not imgs:
            raise FileNotFoundError(f"No images found under {IMAGES_DIR}")
        return random.choice(imgs)
    p = Path(arg)
    if p.is_file():
        return p
    stem = p.stem if p.suffix else str(arg)
    matches = list(IMAGES_DIR.rglob(f"{stem}.*"))
    if not matches:
        raise FileNotFoundError(f"Could not find image for '{arg}' under {IMAGES_DIR}")
    return matches[0]


def find_annotation(img_path):
    xml_path = ANNOT_DIR / img_path.parent.name / f"{img_path.stem}.xml"
    if not xml_path.is_file():
        raise FileNotFoundError(f"Annotation missing: {xml_path}")
    return xml_path


def find_clean(img_path):
    layout_id = img_path.stem.split("_")[0]
    candidates = list(CLEAN_DIR.glob(f"{layout_id}.*"))
    if not candidates:
        raise FileNotFoundError(f"No clean example for layout {layout_id} in {CLEAN_DIR}")
    return candidates[0]


def parse_boxes(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes = []
    for obj in root.findall("object"):
        name = obj.findtext("name", default="?")
        b = obj.find("bndbox")
        boxes.append((
            name,
            int(b.findtext("xmin")), int(b.findtext("ymin")),
            int(b.findtext("xmax")), int(b.findtext("ymax")),
        ))
    return boxes

In [ ]:
def show(sample=None):
    flaw_path = resolve_sample(sample)
    xml_path = find_annotation(flaw_path)
    clean_path = find_clean(flaw_path)
    boxes = parse_boxes(xml_path)

    flaw_img = Image.open(flaw_path)
    clean_img = Image.open(clean_path)

    fig, (ax_clean, ax_flaw) = plt.subplots(1, 2, figsize=(16, 7))
    ax_clean.imshow(clean_img)
    ax_clean.set_title(f"Clean reference \u2014 {clean_path.name}")
    ax_clean.axis("off")

    ax_flaw.imshow(flaw_img)
    plural = "es" if len(boxes) != 1 else ""
    ax_flaw.set_title(f"{flaw_path.parent.name} \u2014 {flaw_path.name}  ({len(boxes)} box{plural})")
    ax_flaw.axis("off")

    for name, xmin, ymin, xmax, ymax in boxes:
        ax_flaw.add_patch(Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor="red", facecolor="none",
        ))
        ax_flaw.text(
            xmin, max(ymin - 8, 0), name,
            color="white", fontsize=8,
            bbox=dict(facecolor="red", edgecolor="none", pad=1),
        )

    plt.tight_layout()
    plt.show()
    return flaw_path, boxes


show(SAMPLE)

## Browse interactively

Re-run the cell below to step through random samples, or call `show("01_missing_hole_03")` with a specific stem.

In [ ]:
show()  # random sample each run